<a href="https://colab.research.google.com/github/brandonrecoder8/AAI2025/blob/main/Chaining_Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Prompt Engineering Assignment: Customer Support Workflow Assistant

### Exercise 1: Ready-To-Use Prompt Chain

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [20]:
# Import the Python SDK
from google import genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

### Customer Message

In [21]:
customer_message = "My wireless headphones stopped connecting after I updated my phone. I need them for a flight tomorrow."
print(f"Customer Message: {customer_message}")

Customer Message: My wireless headphones stopped connecting after I updated my phone. I need them for a flight tomorrow.


### System Prompt

In [22]:
system_prompt = """You are a customer-support workflow assistant for a consumer electronics company.
Be professional, empathetic, concise, and safety-conscious.
Do not invent company policies, refunds, warranties, or technical facts.
When information is missing, identify it clearly rather than assuming it.
Return only the format requested in each step."""
print(f"System Prompt: {system_prompt}")

System Prompt: You are a customer-support workflow assistant for a consumer electronics company.
Be professional, empathetic, concise, and safety-conscious.
Do not invent company policies, refunds, warranties, or technical facts.
When information is missing, identify it clearly rather than assuming it.
Return only the format requested in each step.


### Step 1: Classify the issue

In [23]:
step_1_prompt = f"""Customer message:
\"{customer_message}\"

Task:
Classify this support request.

Return valid JSON only with these fields:
{{
  \"category\": \"one of: connectivity, battery, delivery, billing, return, account, other\",
  \"urgency\": \"low, medium, or high\",
  \"sentiment\": \"positive, neutral, frustrated, or urgent\",
  \"summary\": \"maximum 25 words\",
  \"reason\": \"one short sentence explaining the classification\"
}}

Rules:
- Use only information stated in the customer message.
- Do not propose a solution yet.
- Mark urgency as high only if the customer gives a time-sensitive reason or cannot use an essential purchased product."""

response_step_1 = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=[
        system_prompt,
        step_1_prompt
    ]
)

step_1_output = response_step_1.text
print(f"Step 1 Output:\n{step_1_output}")

Step 1 Output:
{
  "category": "connectivity",
  "urgency": "high",
  "sentiment": "urgent",
  "summary": "Wireless headphones stopped connecting after a phone update and are needed for a flight tomorrow.",
  "reason": "Urgency is high because the customer states they need the device for a flight tomorrow."
}


### Step 2: Identify missing details

In [24]:
import json

# Ensure step_1_output is a valid JSON string before embedding, or handle potential errors.
# For this step, we'll assume a valid JSON output from step 1 for prompt construction.
# If step_1_output was an error, this will still create the prompt string correctly,
# but the LLM call will depend on step_1_output being an actual response.
step_2_prompt = f"""Original customer message:
\"{customer_message}\"

Issue classification from Step 1:
{step_1_output}

Task:
Identify the minimum information needed before troubleshooting the issue.

Return valid JSON only:
{{
  \"missing_information\": [
    \"up to 4 concise questions or information requests\"
  ],
  \"information_already_known\": [
    \"facts stated by the customer\"
  ],
  \"do_not_ask_again\": [
    \"facts already provided\"
  ]
}}

Rules:
- Ask only questions relevant to a connectivity issue.
- Do not ask for personal data, payment information, or unnecessary details.
- Do not repeat information already contained in the original message.
- Keep each question under 18 words."""

response_step_2 = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=[
        system_prompt,
        step_2_prompt
    ]
)

step_2_output = response_step_2.text
print(f"Step 2 Output:\n{step_2_output}")

Step 2 Output:
{
  "missing_information": [
    "What is the brand and model of your wireless headphones?",
    "What is the operating system and version of your phone?",
    "Have you tried forgetting the headphones in your phone's Bluetooth settings and reconnecting?",
    "Do the headphones connect successfully to any other devices?"
  ],
  "information_already_known": [
    "Wireless headphones stopped connecting after a phone update.",
    "The customer needs the headphones for a flight tomorrow."
  ],
  "do_not_ask_again": [
    "Issue started after a phone update",
    "Flight tomorrow"
  ]
}


### Step 3: Draft a customer response

In [25]:
step_3_prompt = f"""Customer message:
\"{customer_message}\"

Classification from Step 1:
{step_1_output}

Missing-information analysis from Step 2:
{step_2_output}

Task:
Write a customer-support reply that acknowledges the urgency, gives safe first troubleshooting steps, and asks only the needed follow-up questions.

Requirements:
- Tone: empathetic, calm, and professional.
- Length: 120–160 words.
- Start with one sentence acknowledging the flight deadline.
- Give exactly 3 numbered troubleshooting steps.
- Ask no more than 2 follow-up questions.
- Do not promise a replacement, refund, or resolution.
- Do not mention internal systems, policies, or that you are an AI.
- End with a brief invitation to reply with the requested details."""

response_step_3 = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=[
        system_prompt,
        step_3_prompt
    ]
)

step_3_output = response_step_3.text
print(f"Step 3 Output:\n{step_3_output}")

Step 3 Output:
I understand how stressful it is to have your wireless headphones stop working right before a flight tomorrow. To help resolve this connection issue quickly, please try the following troubleshooting steps: First, turn your phone's Bluetooth completely off for ten seconds, then turn it back on. Second, go into your phone's Bluetooth settings, find your headphones, and select "Forget This Device" or "Unpair." Third, restart your phone and attempt to put your headphones back into pairing mode to reconnect them. 

To help us investigate further if these steps do not work, could you please let us know the exact brand and model of your wireless headphones, along with the operating system and version of your phone? Please reply with these details so we can assist you further.


### Step 4: Escalation decision

In [26]:
step_4_prompt = f"""Customer message:
\"{customer_message}\"

Issue classification:
{step_1_output}

Missing-information analysis:
{step_2_output}

Draft response:
{step_3_output}

Task:
Decide whether this case should be escalated to a human support specialist.

Return valid JSON only:
{{
  \"escalate\": true or false,
  \"priority\": \"standard, expedited, or urgent\",
  \"reason\": \"maximum 35 words\",
  \"next_action\": \"one concise action\"
}}

Escalate if:
- There is a safety concern.
- The customer reports repeated failure after basic troubleshooting.
- The issue involves payment, account security, legal threats, or accessibility needs.
- A time-sensitive problem remains unresolved after troubleshooting.

Do not escalate solely because the customer is frustrated."""

response_step_4 = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=[
        system_prompt,
        step_4_prompt
    ]
)

step_4_output = response_step_4.text
print(f"Step 4 Output:\n{step_4_output}")

Step 4 Output:
{
  "escalate": false,
  "priority": "standard",
  "reason": "Basic troubleshooting has not yet been attempted by the customer, and no safety concerns, payment issues, or repeated failures are reported.",
  "next_action": "Wait for the customer to reply with the requested troubleshooting results and missing device information."
}


### Required Iteration Evidence

This table illustrates how the constraints and rules applied in each step contribute to a more refined and effective customer support workflow.

| Version | Problem | Improvement |
| :------ | :------ | :---------- |
| **Before** | Step 1 classification could be too generic or lack specific details. | Added specific categories, urgency levels, sentiment, summary, and a reason for classification, ensuring structured output. |
| **Before** | Step 2 might ask for information already provided or irrelevant details, leading to redundant questions. | Introduced rules to only ask questions relevant to a connectivity issue, avoid personal data, and explicitly `do_not_ask_again` for facts already known, leading to more focused and efficient information gathering. |
| **Before** | Step 3 responses could be inconsistent in tone, length, or content, potentially overwhelming the customer or failing to address urgency. | Imposed strict requirements on tone (empathetic, calm, professional), length (120–160 words), and structure (acknowledging deadline, exactly 3 troubleshooting steps, no more than 2 follow-up questions), ensuring a clear, concise, and helpful response. |
| **Before** | Step 4 escalation decisions might be subjective or inconsistent. | Defined clear escalation criteria (safety, repeated failure, payment, security, time-sensitive unresolved issues) and exclusion rules (not solely for frustration), leading to objective and consistent escalation protocols. |
| **After** | **Overall** | The structured prompt chain ensures a more concise, consistent, and customer-friendly experience, while also streamlining the agent's workflow by guiding them through necessary information gathering and decision-making processes. |


## Document Checklist: Prompt Engineering Assignment

### Exercise 1 (Prompt Chaining)

#### Tool(s) used
*   Google Colab
*   Google Gemini API (via `google-generativeai` Python SDK)
*   Python

#### Prompts listed clearly by step

**Step 1: Classify the issue**

**Prompt:**
```
Customer message:
"{{customer_message}}"

Task:
Classify this support request.

Return valid JSON only with these fields:
{
  "category": "one of: connectivity, battery, delivery, billing, return, account, other",
  "urgency": "low, medium, or high",
  "sentiment": "positive, neutral, frustrated, or urgent",
  "summary": "maximum 25 words",
  "reason": "one short sentence explaining the classification"
}

Rules:
- Use only information stated in the customer message.
- Do not propose a solution yet.
- Mark urgency as high only if the customer gives a time-sensitive reason or cannot use an essential purchased product.
```

**Step 2: Identify missing details**

**Prompt:**
```
Original customer message:
"{{customer_message}}"

Issue classification from Step 1:
{{step_1_output}}

Task:
Identify the minimum information needed before troubleshooting the issue.

Return valid JSON only:
{
  "missing_information": [
    "up to 4 concise questions or information requests"
  ],
  "information_already_known": [
    "facts stated by the customer"
  ],
  "do_not_ask_again": [
    "facts already provided"
  ]
}

Rules:
- Ask only questions relevant to a connectivity issue.
- Do not ask for personal data, payment information, or unnecessary details.
- Do not repeat information already contained in the original message.
- Keep each question under 18 words.
```

**Step 3: Draft a customer response**

**Prompt:**
```
Customer message:
"{{customer_message}}"

Classification from Step 1:
{{step_1_output}}

Missing-information analysis from Step 2:
{{step_2_output}}

Task:
Write a customer-support reply that acknowledges the urgency, gives safe first troubleshooting steps, and asks only the needed follow-up questions.

Requirements:
- Tone: empathetic, calm, and professional.
- Length: 120–160 words.
- Start with one sentence acknowledging the flight deadline.
- Give exactly 3 numbered troubleshooting steps.
- Ask no more than 2 follow-up questions.
- Do not promise a replacement, refund, or resolution.
- Do not mention internal systems, policies, or that you are an AI.
- End with a brief invitation to reply with the requested details.
```

**Step 4: Escalation decision**

**Prompt:**
```
Customer message:
"{{customer_message}}"

Issue classification:
{{step_1_output}}

Missing-information analysis:
{{step_2_output}}

Draft response:
{{step_3_output}}

Task:
Decide whether this case should be escalated to a human support specialist.

Return valid JSON only:
{
  "escalate": true or false,
  "priority": "standard, expedited, or urgent",
  "reason": "maximum 35 words",
  "next_action": "one concise action"
}

Escalate if:
- There is a safety concern.
- The customer reports repeated failure after basic troubleshooting.
- The issue involves payment, account security, legal threats, or accessibility needs.
- A time-sensitive problem remains unresolved after troubleshooting.

Do not escalate solely because the customer is frustrated.
```

#### A short note describing what each step does and how it uses prior output

*   **Step 1: Classify the issue:** This step takes the raw `customer_message` and classifies it into a structured JSON format, detailing category, urgency, sentiment, a summary, and the reason. It uses the `system_prompt` for overall guidance.
*   **Step 2: Identify missing details:** This step uses the `customer_message` and the `step_1_output` (classification) to intelligently determine what additional information is needed from the customer. It generates a list of concise questions, identifies known facts, and avoids asking redundant questions.
*   **Step 3: Draft a customer response:** This step leverages the `customer_message`, `step_1_output` (classification), and `step_2_output` (missing information) to craft an empathetic and professional customer service reply. The response includes an acknowledgment of urgency, provides initial troubleshooting steps, and asks only the necessary follow-up questions identified in Step 2.
*   **Step 4: Escalation decision:** This final step uses the `customer_message`, `step_1_output`, `step_2_output`, and the `step_3_output` (draft response) to make an informed decision on whether the case requires escalation to a human agent. It provides a structured JSON output with an escalation flag, priority, reason, and the next action, based on predefined criteria.

#### Final successful output included

**Step 1 Output:**
```json
{kernel_variable: step_1_output}
```

**Step 2 Output:**
```json
{kernel_variable: step_2_output}
```

**Step 3 Output:**
```
{kernel_variable: step_3_output}
```

**Step 4 Output:**
```json
{kernel_variable: step_4_output}
```